# Nairobi flood susceptibility — staged review v5.2

Run one stage at a time and approve each checkpoint. This is a relative screening workflow, not a flood-depth model or forecast. Restart the kernel before Stage 0.

## Stage 0 — Environment and version

In [ ]:
# Run this first in a freshly restarted kernel.
import importlib.util
import os
from pathlib import Path

rasterio_spec = importlib.util.find_spec('rasterio')
proj_dir = Path(rasterio_spec.origin).parent / 'proj_data'
assert (proj_dir / 'proj.db').exists(), f'Bundled proj.db not found: {proj_dir}'
os.environ['PROJ_DATA'] = str(proj_dir)
os.environ['PROJ_LIB'] = str(proj_dir)
print('PROJ database selected:', proj_dir)

In [ ]:
import importlib
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import rasterio
import rioxarray
import xarray as xr
from shapely.geometry import Point, box

import flood_risk_planetary as frp
frp = importlib.reload(frp)
# The module version is reported below; a stale notebook label must not block execution.
print('Pipeline:', frp.PIPELINE_VERSION)
print('Active PROJ database:', pyproj.datadir.get_data_dir())
print('Rasterio:', rasterio.__version__)

## Stage 1 — Cache-first loading and review of all analysis inputs

Stage 1 first looks for the complete `raw_data/` bundle. If present, it restores the saved configuration, source inventory, aligned rasters, AOI/zones, drainage, buildings, river reaches, corridor, and distance bands without calling Planetary Computer or Overpass. If absent, it downloads each source once, builds the review layers once, and writes the complete cache automatically.


In [ ]:
RAW_DATA_DIR = Path('raw_data')
USE_RAW_DATA_CACHE = True
CACHE_AVAILABLE = USE_RAW_DATA_CACHE and frp.analysis_input_cache_ready(RAW_DATA_DIR)
if CACHE_AVAILABLE:
    cached_config = frp.PCModelConfig.from_json(RAW_DATA_DIR / 'config.json')
    CACHE_AVAILABLE = cached_config.drainage_corridor_m == 125

config = frp.PCModelConfig(
    aoi_path=None,  # used only when creating a new cache
    resolution_m=30,
    reporting_resolution_m=90,
    hydrology_buffer_m=5000,
    stream_threshold_km2=1.0,
    drainage_corridor_m=125,
    corridor_hand_max_m=5.0,
    corridor_hazard_threshold=0.60,
)
config.validate()
print('Input mode:', 'local raw_data cache' if CACHE_AVAILABLE else 'rebuild 125 m corridor cache')

In [ ]:
if CACHE_AVAILABLE:
    cached_inputs = frp.load_analysis_input_cache(RAW_DATA_DIR)
    config = cached_inputs.config
    sources = cached_inputs.sources
    raw = cached_inputs.raw
    aoi = cached_inputs.aoi
    neighborhoods = cached_inputs.neighborhoods
    zones = cached_inputs.zones
    mapped_drainage = cached_inputs.mapped_drainage
    osm_buildings = cached_inputs.buildings
    river_reaches_red = cached_inputs.river_reaches
    river_corridor_500m = cached_inputs.river_corridor
    buffer_intervals = cached_inputs.buffer_intervals
    print('Loaded complete local input cache:', cached_inputs.manifest_path.resolve())
else:
    aoi, neighborhoods, zones = frp.make_review_geometries(config)
    sources = frp.discover_sources(config)
    print(f'Discovered {len(sources.inventory)} source records for the first-run cache.')

display(frp.aoi_review_table(aoi, config).round(3))


In [ ]:
if CACHE_AVAILABLE:
    print('Reusing aligned rasters from raw_data; no Planetary Computer request made.')
else:
    raw = frp.load_raw_layers(sources, config)

raw_stats = pd.DataFrame([
    {
        'layer': band,
        'min': float(raw[band].min()),
        'mean': float(raw[band].mean()),
        'max': float(raw[band].max()),
        'missing': int(raw[band].isnull().sum()),
        'missing_pct': float(raw[band].isnull().mean() * 100),
    }
    for band in raw.data_vars
])
print('Processing grid:', dict(raw.sizes), 'CRS:', raw.rio.crs)


In [ ]:
BUFFER_DISTANCES_M = (31.0, 62.5, 125.0)
BUFFER_COLORS = frp.RIVER_BUFFER_COLORS

# These reviewed files replace the former broad waterways/context inputs.
RIVER_FILE = RAW_DATA_DIR / 'vectors' / 'river.geojson'
DAMS_FILE = RAW_DATA_DIR / 'vectors' / 'dams.geojson'
mapped_drainage = frp.load_and_clip_local_waterways(RIVER_FILE, aoi)
mapped_dams = frp.load_and_clip_local_dams(DAMS_FILE, aoi)
river_reaches_red, river_corridor_500m = frp.river_reaches_in_zones(
    mapped_drainage, zones, config
)
buffer_intervals = frp.make_river_buffer_intervals(
    mapped_drainage, zones, config, BUFFER_DISTANCES_M
)

if not CACHE_AVAILABLE:
    BUILDINGS_CACHE = Path('data/osm_buildings_river_corridor_500m.geojson')
    if BUILDINGS_CACHE.exists():
        osm_buildings = gpd.read_file(BUILDINGS_CACHE)
    else:
        osm_buildings = frp.fetch_osm_buildings(river_corridor_500m)
        if not osm_buildings.empty:
            BUILDINGS_CACHE.parent.mkdir(parents=True, exist_ok=True)
            osm_buildings.to_file(BUILDINGS_CACHE, driver='GeoJSON')

    raw_cache_paths = frp.export_analysis_input_cache(
        raw, sources, config, aoi, neighborhoods, zones,
        mapped_drainage, osm_buildings, river_reaches_red,
        river_corridor_500m, buffer_intervals, output_dir=RAW_DATA_DIR,
    )
    print(f'Created complete raw-data cache with {len(raw_cache_paths)} files:', RAW_DATA_DIR.resolve())
else:
    print('Raster, building, and review-area inputs restored from the local cache.')
    print('Reviewed rivers and dams reloaded from river.geojson and dams.geojson.')

input_map = folium.Map(location=[-1.30, 36.86], zoom_start=11, tiles=None, prefer_canvas=True)
frp.add_review_basemaps(input_map)
frp.add_folium_raster_layer(input_map, raw.elevation, name='Elevation / terrain input', cmap='terrain', opacity=0.60, show=False)
frp.add_folium_raster_layer(input_map, raw.worldcover, name='ESA WorldCover input', cmap='tab20', opacity=0.58, show=False, vmin=0, vmax=100, nearest=True)
frp.add_folium_raster_layer(input_map, raw.water_occurrence, name='Surface-water occurrence input', cmap='Blues', opacity=0.65, show=False, vmin=0, vmax=100)
frp.add_folium_raster_layer(input_map, raw.extreme_rainfall_raw, name='Rainfall input', cmap='YlGnBu', opacity=0.62, show=False)

processing = gpd.GeoDataFrame(geometry=[box(*frp.processing_bbox(config))], crs=4326)
folium.GeoJson(processing, name='Hydrology processing extent', style_function=lambda _: {'color':'orange','weight':2,'fillOpacity':0.01}).add_to(input_map)
folium.GeoJson(frp.folium_safe_geodataframe(aoi, ('name',)).to_crs(4326), name='Reporting AOI', style_function=lambda _: {'color':'black','weight':3,'fillOpacity':0.03}).add_to(input_map)
folium.GeoJson(frp.folium_safe_geodataframe(zones, ('name','zone')).to_crs(4326), name='Inspection zones', style_function=lambda _: {'color':'cyan','weight':1,'fillOpacity':0.04}).add_to(input_map)

dam_fields = tuple(field for field in ('display_name','waterbody_type','model_source') if field in mapped_dams.columns)
dam_map_layer = frp.folium_safe_geodataframe(mapped_dams, dam_fields).to_crs(4326)
folium.GeoJson(dam_map_layer, name=f'Reviewed dams and water bodies ({len(mapped_dams):,})', style_function=lambda _: {'color':'#075985','weight':1.5,'fillColor':'#38bdf8','fillOpacity':0.40}, tooltip=folium.GeoJsonTooltip(fields=list(dam_fields)) if dam_fields else None).add_to(input_map)

drainage_fields = tuple(field for field in ('display_name','waterway','inspection_area','model_source') if field in river_reaches_red.columns)
drainage_map_layer = frp.folium_safe_geodataframe(river_reaches_red, drainage_fields).to_crs(4326)
folium.GeoJson(drainage_map_layer, name=f'Reviewed rivers and streams in six study areas ({len(river_reaches_red):,})', style_function=lambda _: {'color':'#1685d1','weight':4,'opacity':0.95}, tooltip=folium.GeoJsonTooltip(fields=list(drainage_fields)) if drainage_fields else None).add_to(input_map)

river_polygon_layer = frp.folium_safe_geodataframe(river_corridor_500m, ('buffer_m_each_side',)).to_crs(4326)
folium.GeoJson(river_polygon_layer, name='River/stream corridor polygon', show=True, style_function=lambda _: {'color':'#1685d1','weight':2,'fillColor':'#1685d1','fillOpacity':0.10}).add_to(input_map)

interval_fields = tuple(field for field in ('inspection_area','interval_label','distance_to_m','area_ha') if field in buffer_intervals.columns)
interval_map_layer = frp.folium_safe_geodataframe(buffer_intervals, interval_fields).to_crs(4326)
def interval_style(feature):
    upper = float(feature['properties']['distance_to_m'])
    color = BUFFER_COLORS[upper]
    return {'color':color,'weight':1,'fillColor':color,'fillOpacity':0.28}
folium.GeoJson(interval_map_layer, name='Exclusive river-distance intervals', show=False, style_function=interval_style, tooltip=folium.GeoJsonTooltip(fields=list(interval_fields))).add_to(input_map)

if not osm_buildings.empty:
    buildings_map_layer = frp.folium_safe_geodataframe(osm_buildings).to_crs(4326)
    folium.GeoJson(buildings_map_layer, name=f'OSM buildings ({len(osm_buildings):,})', show=False, style_function=lambda _: {'color':'#555','weight':0.5,'fillColor':'#888','fillOpacity':0.25}).add_to(input_map)

frp.add_inspection_markers(input_map, neighborhoods)
folium.LayerControl(collapsed=False).add_to(input_map)
print(f'Loaded {len(mapped_drainage):,} reviewed river/stream lines and {len(mapped_dams):,} dam/water-body polygons; modelling {len(river_reaches_red):,} line parts in the six study areas with {len(osm_buildings):,} buildings.')
input_map


**How to read this output:** the input-review map is a spatial check of every source before modelling. Toggle a layer on, compare it with the basemap and inspection zones, and use its key where available. In the WorldCover layer, the pixel values are ESA class codes, not risk scores:

| Code | Class |
| ---: | --- |
| 10 | Tree cover |
| 20 | Shrubland |
| 30 | Grassland |
| 40 | Cropland |
| 50 | Built-up |
| 60 | Bare / sparse vegetation |
| 70 | Snow and ice |
| 80 | Permanent water bodies |
| 90 | Herbaceous wetland |
| 95 | Mangroves |
| 100 | Moss and lichen |

Class 50 supplies the built-up indicator and the land-cover classes supply the runoff lookup. A class code does not mean higher or lower flood risk by itself.

**Checkpoint:** approve the reporting polygon, upstream buffer, drainage alignment, building coverage, target points, terrain grid, reporting scale, and source suitability. The cache manifest makes this reviewed input state reproducible.

## Stage 2 — Source inventory and provenance

Stage 1 already discovered and loaded these sources. This stage reviews the retained inventory without another search or download.


In [ ]:
display(sources.inventory)


**Checkpoint:** verify collection, asset, period, item IDs, retrieval time, and buffered search extent. TerraClimate remains a coarse temporary climatology.

## Stage 3 — Raw aligned-input QA

The raw dataset was loaded once in Stage 1. These panels and statistics reuse it without another Planetary Computer request.


In [ ]:
REPORT_ASSET_DIR = Path('exports/report_assets')
REPORT_ASSET_DIR.mkdir(parents=True, exist_ok=True)
display(raw_stats.round(3))
raw_panels = [
    ('elevation','terrain'), ('worldcover','tab20'),
    ('water_occurrence','Blues'), ('extreme_rainfall_raw','YlGnBu'),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for ax, (band, cmap) in zip(axes.flat, raw_panels):
    raw[band].plot(ax=ax, cmap=cmap, robust=True)
    zones.boundary.plot(ax=ax, color='magenta', linewidth=0.8)
    ax.set_title(band.replace('_',' ').title())
    ax.set_axis_off()
fig.savefig(REPORT_ASSET_DIR / 'figure_01_raw_inputs.png', dpi=180, bbox_inches='tight')
plt.show()


**How to read this output:** the statistics table checks numeric ranges and missing cells; Figure 1 checks spatial alignment. Read the WorldCover panel using the ESA class key in the Stage 1 explanation. Do not infer 30 m rainfall detail from the interpolated rainfall display.

**Checkpoint:** reject missing strips, implausible ranges, or misalignment before continuing.

## Stage 4 — Reviewed rivers, dams, DEM conditioning, and routed hydrology

The analysis uses the reviewed `river.geojson` centre-lines and `dams.geojson` water-body polygons loaded in Stage 1. Both are rasterized as reference drainage for HAND and water proximity. River lines alone define the 125 m corridor and building-distance bands. Review topology, names, alignment, completeness, licence, and source date before interpreting results.

In [ ]:
# Reuse the reviewed river and dam layers loaded in Stage 1.
assert 'mapped_drainage' in globals() and not mapped_drainage.empty
assert 'mapped_dams' in globals() and not mapped_dams.empty
print(f'Reusing {len(mapped_drainage):,} reviewed rivers/streams and {len(mapped_dams):,} dam/water-body polygons from Stage 1.')
waterway_preview_fields = [
    field for field in ('display_name', 'waterway', 'intermittent', 'osm_id')
    if field in mapped_drainage.columns
]
display(mapped_drainage[waterway_preview_fields].head(20))
derived = frp.derive_predictors(raw, config, mapped_drainage=mapped_drainage, mapped_dams=mapped_dams)
derived_stats = pd.DataFrame([
    {
        'layer': band,
        'min': float(derived[band].min()),
        'mean': float(derived[band].mean()),
        'max': float(derived[band].max()),
        'missing': int(derived[band].isnull().sum()),
    }
    for band in derived.data_vars
])
derived_stats.round(3)


In [ ]:
hydrology_panels = [
    ('conditioned_elevation','terrain'), ('depression_storage_raw','Blues'),
    ('flow_accumulation_km2','viridis'), ('modeled_stream','Blues'),
    ('topographic_wetness_raw','YlGnBu'), ('hand_m','magma'),
    ('distance_to_drainage_m','cividis'), ('drainage_proximity','Blues'),
]
fig, axes = plt.subplots(2, 4, figsize=(20, 10), constrained_layout=True)
for ax, (band, cmap) in zip(axes.flat, hydrology_panels):
    derived[band].plot(ax=ax, cmap=cmap, robust=True)
    aoi.boundary.plot(ax=ax, color='red', linewidth=0.8)
    ax.set_title(band.replace('_',' ').title())
    ax.set_axis_off()
fig.savefig(REPORT_ASSET_DIR / 'figure_02_hydrology.png', dpi=180, bbox_inches='tight')
plt.show()


**How to read this output:** Table 3 reports numerical ranges for derived layers; Figure 2 checks whether terrain, accumulation, modelled streams, HAND, and drainage proximity form plausible connected patterns. These are hydrologic diagnostics, not independent flood observations.

**Checkpoint:** mapped and modelled streams should follow valleys; accumulation should increase downstream; HAND should be low beside drainage; depression storage should not reproduce the old mean-filter pattern. Treat absent OpenStreetMap features as unknown coverage, not proof that no drain exists.

## Stage 5 — Normalized predictors and redundancy check

In [ ]:
normalized = frp.normalize_predictors(raw, derived, config)
normalized_stats = pd.DataFrame([
    {'predictor':band, 'min':float(normalized[band].min()),
     'mean':float(normalized[band].mean()), 'max':float(normalized[band].max())}
    for band in normalized.data_vars
])
display(normalized_stats.round(3))
correlation = frp.predictor_correlation(normalized, config)
display(correlation.round(2).style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1))

corr_fig, corr_ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
image = corr_ax.imshow(correlation.values, cmap='RdBu_r', vmin=-1, vmax=1)
corr_ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha='right')
corr_ax.set_yticks(range(len(correlation.index)), correlation.index)
for row in range(len(correlation.index)):
    for column in range(len(correlation.columns)):
        corr_ax.text(column, row, f'{correlation.iloc[row, column]:.2f}', ha='center', va='center', fontsize=8)
corr_ax.set_title('Spearman correlation among normalized predictors')
corr_fig.colorbar(image, ax=corr_ax, label='Spearman correlation')
corr_fig.savefig(REPORT_ASSET_DIR / 'figure_03_predictor_correlation.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
for ax, band in zip(axes.flat, normalized.data_vars):
    normalized[band].plot(ax=ax, cmap='RdYlBu_r', vmin=0, vmax=1)
    aoi.boundary.plot(ax=ax, color='cyan', linewidth=0.7)
    ax.set_title(band.replace('_',' ').title())
    ax.set_axis_off()
fig.savefig(REPORT_ASSET_DIR / 'figure_04_normalized_predictors.png', dpi=180, bbox_inches='tight')
plt.show()


**How to read this output:** the statistics table confirms each normalized predictor is within 0–1; the correlation matrix identifies possible redundancy; Figure 4 shows where each predictor is relatively strong or weak. Correlation colors indicate association, not hazard level.

**Checkpoint:** all normalized predictors must be 0–1. Investigate absolute Spearman correlations above about 0.8 before retaining both predictors.

## Stage 6 — Hazard, strict data mask, and relative classes

In [ ]:
scored,weight_review=frp.score_model(normalized,derived,config)
display(weight_review.round(4))
print('Relative class breaks:',scored.attrs['class_breaks'])
print('Complete-data coverage:',float(scored.complete_data_mask.mean()))
corridors=frp.assess_drainage_corridors(mapped_drainage,zones,derived,scored,config)
display(corridors.summary.round(3))

buffer_building_exposure, buffer_building_summary = (
    frp.assess_candidate_buildings_by_buffer_interval(
        osm_buildings, buffer_intervals, corridors.layers.candidate_affected_corridor
    )
)
display(buffer_building_summary.round(1))
print('Potentially affected means the building intersects the screening corridor; it is not confirmed flood damage.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
scored.data_coverage.plot(ax=axes[0], cmap='viridis', vmin=0, vmax=1)
scored.hazard_score.plot(ax=axes[1], cmap='RdYlBu_r', vmin=0, vmax=1)
scored.planning_priority_score.plot(ax=axes[2], cmap='RdYlBu_r', vmin=0, vmax=1)
scored.hazard_class.where(scored.hazard_class > 0).plot(ax=axes[3], cmap='RdYlBu_r', vmin=1, vmax=5)
for ax, title in zip(axes, ['Data coverage','Relative hazard','Built-density planning proxy','Relative hazard quintile']):
    aoi.boundary.plot(ax=ax, color='cyan', linewidth=0.8)
    ax.set_title(title)
    ax.set_axis_off()
fig.savefig(REPORT_ASSET_DIR / 'figure_05_hazard_and_priority.png', dpi=180, bbox_inches='tight')
plt.show()

corridor_fig = frp.plot_reviewed_hydrography_screening(
    scored.hazard_score, corridors.layers.candidate_affected_corridor,
    mapped_drainage, mapped_dams, zones,
)
corridor_fig.savefig(REPORT_ASSET_DIR / 'figure_06_candidate_corridors.png', dpi=180, bbox_inches='tight')
plt.show()


**How to read this output:** the weight table explains each predictor's contribution; the corridor table summarizes river screening by inspection area; Figure 5 compares coverage, hazard, planning proxy, and relative classes; and Figure 6 checks candidate corridors against reviewed rivers. Dam/water-body polygons contribute to HAND reference drainage and water proximity but do not receive river buffer bands. Masked cells were not scored; they are not evidence of low hazard.

**Checkpoint:** incomplete cells must remain masked. Review river topology and dam polygon alignment. Candidate corridors require river proximity, HAND <= 5 m, and hazard >= 0.60; they are not probabilities, depths, return periods, or hydraulic inundation extents.

## Stage 6b — Optional Sentinel-1 extent in the Stage 1 river corridors

The drainage reaches, five exclusive distance bands, and OSM buildings were loaded once and reviewed in Stage 1. This optional stage reuses them; it does not rebuild the corridor map or download the building inventory again.


In [ ]:
assert all(name in globals() for name in ('river_reaches_red','river_corridor_500m','buffer_intervals','osm_buildings'))
print(f'Reusing Stage 1 inputs: {len(river_reaches_red):,} river reaches, {len(buffer_intervals):,} interval polygons, and {len(osm_buildings):,} buildings.')
display(buffer_intervals[['inspection_area','interval_label','area_ha']].round(3))


In [ ]:
HISTORICAL_WINDOWS = {
    '2015_Event': '2015-10-01/2015-12-31',
    '2019_Event': '2019-10-01/2019-12-31',
    '2023_Event': '2023-10-01/2023-12-31',
}
RUN_SENTINEL1_EXTENT = False  # Requires Planetary Computer RTC asset access and cloud compute
if RUN_SENTINEL1_EXTENT:
    extent_result = frp.analyze_sentinel1_river_extents(
        mapped_drainage, zones, config, HISTORICAL_WINDOWS,
        buildings=osm_buildings, water_threshold_db=-16.0, resolution_m=20,
        historical_labels=('2015_Event','2019_Event','2023_Event'),
        prediction_year=2026, prediction_label='2026_Prediction',
    )
    display(extent_result.summary.round(3))
    available = list(extent_result.masks.data_vars)
    fig, axes = plt.subplots(1, len(available), figsize=(6*len(available), 6), squeeze=False, constrained_layout=True)
    for ax, label in zip(axes.flat, available):
        extent_result.masks[label].where(extent_result.masks[label] == 1).plot(ax=ax, cmap='Blues' if label != '2026_Prediction' else 'Reds', add_colorbar=False)
        river_reaches_red.plot(ax=ax, color='red', linewidth=1.5)
        zones.boundary.plot(ax=ax, color='black', linewidth=0.8)
        ax.set_title(f'{label}: water in 500 m corridor')
        ax.set_axis_off()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    extent_result.historical_max.where(extent_result.historical_max).plot(ax=axes[0], cmap='Blues', add_colorbar=False)
    extent_result.regression.predicted_water.where(extent_result.regression.predicted_water == 1).plot(ax=axes[1], cmap='Reds', add_colorbar=False)
    extent_result.predicted_expansion.where(extent_result.predicted_expansion).plot(ax=axes[2], cmap='Oranges', add_colorbar=False)
    for ax, title in zip(axes, ['Historical maximum (2015/2019/2023)', '2026 regression projection', 'Predicted expansion beyond historical maximum']):
        river_reaches_red.plot(ax=ax, color='red', linewidth=1.2)
        zones.boundary.plot(ax=ax, color='black', linewidth=0.8)
        ax.set_title(title); ax.set_axis_off()
    plt.show()
    display(extent_result.regression[['regression_rmse_db','regression_r2']])

    building_exposure, interval_summary = frp.assess_buildings_by_buffer_interval(
        osm_buildings, buffer_intervals, extent_result.masks
    )
    display(interval_summary.round(3))
    display(interval_summary.pivot_table(index=['inspection_area','interval_label'], columns='event', values='osm_buildings_affected', fill_value=0))
    river_figures = frp.plot_river_buffer_analysis(
        extent_result, river_reaches_red, buffer_intervals, building_exposure, interval_summary
    )
    for figure in river_figures.values():
        display(figure)

    RUN_RIVER_EXPORTS = True
    if RUN_RIVER_EXPORTS:
        river_export_paths = frp.export_river_buffer_analysis(
            extent_result, river_reaches_red, buffer_intervals,
            building_exposure, interval_summary, output_dir='exports/river_corridor'
        )
        print('River-corridor outputs:', *(path.resolve() for path in river_export_paths), sep='\n')
else:
    print('Sentinel-1 stage paused. Set RUN_SENTINEL1_EXTENT=True after reviewing the red reaches, corridor, and OSM buildings.')

**How to read this output:** the interval table and event figures describe water-screening results by exclusive distance band; the building table and charts count mapped footprints intersecting those bands. A 2026 projection is an extrapolation from the historical composites, not an observation or operational forecast.

**Checkpoint:** the three bands are exclusive (0–31, 31–62.5, and 62.5–125 m), so each building is counted once per inspection area in its nearest intersected band. Darker blue means nearer the river. Verify red river alignment and OSM completeness.

## Stage 6c — Check whether neighborhood rankings change when model weights change

In [ ]:
sensitivity = frp.weight_sensitivity(
    normalized, zones, config, simulations=500, concentration=100, seed=42
)
display(sensitivity.round(3))
rank_columns = [column for column in ('rank_best','rank_median','rank_worst') if column in sensitivity.columns]
if rank_columns:
    sensitivity_fig, sensitivity_ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
    sensitivity.set_index('name')[rank_columns].plot(kind='bar', ax=sensitivity_ax)
    sensitivity_ax.set_ylabel('Neighborhood rank')
    sensitivity_ax.set_title('Do neighborhood rankings change when model weights change?')
    sensitivity_ax.invert_yaxis()
    sensitivity_fig.savefig(REPORT_ASSET_DIR / 'figure_07_weight_sensitivity.png', dpi=180, bbox_inches='tight')
    plt.show()


**How to read this output:** Table 8 shows how each neighborhood's score and position change when the model weights are changed slightly. Figure 7 makes these changes easy to compare. Use the full best-to-worst range, not only the middle rank.

**Checkpoint:** a large best-to-worst range means the result depends strongly on the chosen weights and should be reported as uncertain.

## Stage 6d — Wilson Airport–South C terrain catchment check

This stage delineates the DEM contributing area whose terrain-routed flow reaches a pour point near South C. The South C marker is snapped to the largest flow-accumulation cell within 1 km; no mapped river is required. An approximate Wilson Airport reference point is checked against the resulting upstream area. Replace it with a reviewed airport boundary if airport-only area or runoff is required.

Catchment area is reported directly. Runoff volume requires a specified storm depth and runoff coefficient; it is not inferred from catchment area alone.

In [ ]:
SOUTH_C_SNAP_RADIUS_M = 1000
DESIGN_STORM_MM = None       # Set only from a reviewed event/IDF source
RUNOFF_COEFFICIENT = None    # Set from a defensible land-surface method

south_c_catchment = frp.delineate_terrain_catchment(
    raw, neighborhoods, config,
    location_name='South C',
    snap_radius_m=SOUTH_C_SNAP_RADIUS_M,
    scored=scored,
)

# Approximate reference point for the airfield, used only to check membership.
wilson_airport_reference = gpd.GeoDataFrame(
    [{'name': 'Wilson Airport (approximate reference)'}],
    geometry=[Point(36.8148, -1.3211)], crs='EPSG:4326'
).to_crs(config.crs)
airport_reference_in_catchment = south_c_catchment.catchment.geometry.iloc[0].covers(
    wilson_airport_reference.geometry.iloc[0]
)
south_c_summary = south_c_catchment.summary.copy()
south_c_summary['wilson_airport_reference_in_catchment'] = airport_reference_in_catchment

# South C has no mapped river, so screen buildings using its terrain drainage area.
SOUTH_C_BUILDINGS_FILE = Path('data/osm_buildings_south_c.geojson')
if SOUTH_C_BUILDINGS_FILE.exists():
    south_c_osm_buildings = gpd.read_file(SOUTH_C_BUILDINGS_FILE)
    print(f'Reusing {len(south_c_osm_buildings):,} cached South C building footprints.')
else:
    south_c_osm_buildings = frp.fetch_osm_buildings(south_c_catchment.catchment)
    south_c_osm_buildings.to_file(SOUTH_C_BUILDINGS_FILE, driver='GeoJSON')
    print(f'Downloaded and cached {len(south_c_osm_buildings):,} South C building footprints.')

south_c_building_exposure, south_c_building_summary = (
    frp.assess_buildings_in_terrain_catchment(
        south_c_osm_buildings, south_c_catchment.catchment, scored.hazard_score,
        location_name='South C', hazard_threshold=config.corridor_hazard_threshold,
    )
)
building_screening_exposure = gpd.GeoDataFrame(
    pd.concat([buffer_building_exposure, south_c_building_exposure], ignore_index=True),
    geometry='geometry', crs=buffer_building_exposure.crs,
)
building_screening_summary = pd.concat(
    [buffer_building_summary, south_c_building_summary], ignore_index=True
)
display(south_c_building_summary.round(1))
if DESIGN_STORM_MM is not None and RUNOFF_COEFFICIENT is not None:
    if DESIGN_STORM_MM <= 0 or not 0 <= RUNOFF_COEFFICIENT <= 1:
        raise ValueError('Storm depth must be positive and runoff coefficient must be within 0–1')
    south_c_summary['scenario_runoff_volume_m3'] = (
        south_c_summary.catchment_area_km2 * DESIGN_STORM_MM * RUNOFF_COEFFICIENT * 1000
    )
display(south_c_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
raw.elevation.plot(ax=axes[0], cmap='terrain')
scored.hazard_score.plot(ax=axes[1], cmap='RdYlBu_r', vmin=0, vmax=1)
for ax, title in zip(axes, ('Terrain contributing area', 'Susceptibility within catchment')):
    south_c_catchment.catchment.boundary.plot(ax=ax, color='magenta', linewidth=2)
    south_c_catchment.pour_point.plot(ax=ax, color='cyan', edgecolor='black', markersize=70)
    wilson_airport_reference.plot(ax=ax, color='yellow', edgecolor='black', markersize=70)
    ax.set_title(title)

print('South C results are ready and will be added to the final map and export bundle in Stage 8.')

**How to read this output:** `catchment_area_km2` is the terrain area contributing to the snapped South C pour point. `wilson_airport_reference_in_catchment=True` confirms that the reference point lies upstream, but an authoritative airport polygon is needed to quantify the airport's own contributing surface. `high_hazard_area_pct` measures overlap with the model's ≥0.60 screening threshold. A true `snap_near_search_limit` flag means the outlet is sensitive to the selected search radius and needs visual confirmation.

**Checkpoint:** confirm the cyan pour point on local drainage/road or culvert data and review the magenta divide. Do not enlarge the search radius automatically: beyond 1 km the algorithm can jump to a different major basin. This is a structural terrain-consistency check using the same DEM as the model—not independent accuracy validation.

## Stage 7a — Optional population, facilities, exposure, and risk

Supply reviewed local inputs. Population must be a density raster (people/km²), not raw cell counts. Without them, only the clearly named built-density planning proxy is retained.

In [ ]:
POPULATION_DENSITY_RASTER=None  # e.g. 'data/worldpop_population_density.tif'
CRITICAL_FACILITIES_FILE=None   # e.g. 'data/critical_facilities.gpkg'
population=frp.load_population_density(POPULATION_DENSITY_RASTER,scored.hazard_score,config) if POPULATION_DENSITY_RASTER else None
facilities=gpd.read_file(CRITICAL_FACILITIES_FILE) if CRITICAL_FACILITIES_FILE else None
exposure=None
if population is not None or facilities is not None:
    exposure=frp.build_exposure_risk(scored.hazard_score,derived.built_density,config,population_density=population,critical_facilities=facilities)
    display(exposure)
else:
    print('Exposure inputs absent: no risk_score created; planning_priority_score remains a proxy.')

## Stage 7b — Independent flood validation

A validation raster must use 1=flooded, 0=confirmed dry, NaN=unknown. Incident points provide positive-case evidence only.

In [ ]:
OBSERVED_FLOOD_MASK=None  # e.g. 'validation/flood_2024_labelled.tif'
INCIDENT_POINTS_FILE=None # e.g. 'validation/geocoded_incidents.gpkg'
if OBSERVED_FLOOD_MASK:
    observed=rioxarray.open_rasterio(OBSERVED_FLOOD_MASK,masked=True).squeeze(drop=True)
    validation_table=frp.validate_flood_mask(scored.hazard_score,observed)
    display(validation_table.round(3))
else:
    print('No labelled flood/dry mask supplied; accuracy is not yet established.')
if INCIDENT_POINTS_FILE:
    incidents=gpd.read_file(INCIDENT_POINTS_FILE)
    incident_scores=frp.sample_incident_points(scored.hazard_score,incidents)
    display(incident_scores.describe())

**How to read this output:** validation metrics quantify agreement with independent flooded and confirmed-dry labels; incident-point scores show model values at reported incidents only. Without confirmed-dry observations, accuracy cannot be established.

**Checkpoint:** do not call the output validated unless independent flooded and confirmed-dry observations have been supplied and reviewed.

## Stage 8 — Combine every completed analysis into one final map and export

This stage combines the flood screening, reviewed rivers and dams, river corridors, buildings, neighborhood areas, and the South C terrain catchment in one interactive map. It also writes one export bundle containing the map, raster layers, tables, and vector files.

**How to read the final outputs:** switch layers on and off in the interactive map. The thicker blue lines are the reviewed rivers and streams, cyan polygons are reviewed dams and mapped water bodies, the translucent blue polygon is the 125 m river screening corridor, and the magenta outline is the land draining through South C. The six points of interest and their buffered study areas are included; analytical outlet/airport points and legacy water-context layers remain omitted. Use the tables for exact values and the GeoTIFF/GeoJSON files for GIS work.

In [ ]:
dashboard_river_streams = river_reaches_red
dashboard_river_polygon = river_corridor_500m

model = frp.assemble_outputs(
    config, sources, raw, derived, normalized, scored,
    exposure=exposure,
    drainage_corridors=corridors.layers,
    drainage_reaches=dashboard_river_streams,
    drainage_corridor_summary=corridors.summary,
    buildings=osm_buildings,
    terrain_catchment_layers=south_c_catchment.layers,
    terrain_catchments=south_c_catchment.catchment,
    terrain_pour_points=None,
    reference_points=None,
    catchment_summary=south_c_summary,
    buffer_building_exposure=building_screening_exposure,
    buffer_building_summary=building_screening_summary,
    mapped_dams=mapped_dams,
)
display(model.summary.round(3))
print('Reporting grid:', dict(model.layers.sizes), model.layers.rio.crs)
print('Variables:', list(model.layers.data_vars))

final_interactive_map = frp.build_final_interactive_map(
    model, buildings=osm_buildings, buffer_intervals=buffer_intervals,
    river_corridor=dashboard_river_polygon, show_point_layers=True,
)
final_interactive_map


In [ ]:
# Reload export helpers so this cell cannot retain stale cache/report logic.
frp = importlib.reload(frp)
CACHE_AVAILABLE = frp.analysis_input_cache_ready(RAW_DATA_DIR)

# Create the reusable cache only when it is genuinely absent at export time.
if not CACHE_AVAILABLE:
    raw_cache_paths = frp.export_analysis_input_cache(
        raw, sources, config, aoi, neighborhoods, zones,
        mapped_drainage, osm_buildings, river_reaches_red,
        river_corridor_500m, buffer_intervals, output_dir=RAW_DATA_DIR,
    )
    print('Created reusable raw-data cache:', *(path.resolve() for path in raw_cache_paths), sep='\n')
else:
    raw_cache_paths = tuple(path for path in RAW_DATA_DIR.rglob('*') if path.is_file())
    print(f'Reusing complete raw-data cache ({len(raw_cache_paths)} files):', RAW_DATA_DIR.resolve())

EXPORT_DIR = Path('exports')
REPORT_ASSET_DIR = EXPORT_DIR / 'report_assets'
REPORT_ASSET_DIR.mkdir(parents=True, exist_ok=True)

# Export every table presented in the staged analysis.
report_tables = {
    'table_01_source_inventory.csv': model.inventory,
    'table_02_raw_input_statistics.csv': raw_stats,
    'table_03_derived_layer_statistics.csv': derived_stats,
    'table_04_normalized_predictor_statistics.csv': normalized_stats,
    'table_05_predictor_correlation.csv': correlation,
    'table_06_model_weights.csv': weight_review,
    'table_07_drainage_corridor_summary.csv': corridors.summary,
    'table_08_weight_sensitivity.csv': sensitivity,
    'table_09_neighborhood_summary.csv': model.summary,
    'table_10_south_c_catchment_summary.csv': south_c_summary,
    'table_11_building_screening_summary.csv': building_screening_summary,
}
if 'validation_table' in globals():
    report_tables['table_10_validation_metrics.csv'] = validation_table
if 'incident_scores' in globals():
    report_tables['table_11_incident_scores.csv'] = incident_scores
if 'interval_summary' in globals():
    report_tables['table_12_river_interval_summary.csv'] = interval_summary
for filename, table in report_tables.items():
    table.to_csv(REPORT_ASSET_DIR / filename, index=True)

input_review_map_path = frp.export_interactive_map(
    input_map, REPORT_ASSET_DIR / 'stage_01_input_review_map.html'
)
paths = list(frp.export_outputs(model, output_dir=EXPORT_DIR))
river_polygon_path = EXPORT_DIR / 'river_corridor_125m.geojson'
dashboard_river_polygon.to_crs('EPSG:4326').to_file(river_polygon_path, driver='GeoJSON')
paths.append(river_polygon_path)
interactive_map_path = frp.export_interactive_map(
    final_interactive_map, EXPORT_DIR / 'nairobi_flood_interactive_map.html'
)
paths.extend([interactive_map_path, input_review_map_path])

river_objects = ('extent_result', 'river_reaches_red', 'buffer_intervals', 'building_exposure', 'interval_summary')
if all(name in globals() for name in river_objects):
    river_export_paths = frp.export_river_buffer_analysis(
        extent_result, river_reaches_red, buffer_intervals,
        building_exposure, interval_summary, output_dir=EXPORT_DIR / 'river_corridor'
    )
    paths.extend(river_export_paths)
else:
    print('River-corridor exports skipped because the optional Sentinel-1 stage was not run.')

# Build a fresh Markdown report from this run's actual tables and attachments.
report_path = frp.export_analysis_report(
    report_tables, config,
    template_path='Nairobi_Flood_Analysis_Report.md',
    output_path=EXPORT_DIR / 'Nairobi_Flood_Analysis_Report.md',
)
paths.append(report_path)

# Refresh the MapLibre dashboard from the same final outputs.
import build_maplibre_dashboard as dashboard_builder
dashboard_builder = importlib.reload(dashboard_builder)
dashboard_builder.build()
print('MapLibre dashboard refreshed. Run: python -m http.server 8000 --directory maplibre_dashboard')
print('Exported model, maps, tables, figures, and refreshed report:',
      *(path.resolve() for path in paths), sep='\n')
